In [115]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import matthews_corrcoef, precision_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier
import pickle
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import re
import itertools
import time
import warnings
warnings.filterwarnings("ignore", module="joblib")
import databento as db
import exchange_calendars as xcals

def close_times():

    # NYSE calendar
    cal = xcals.get_calendar("XNYS")

    # Build schedule for the date range you care about
    #start = "2018-01-01"
    #end = "2030-01-01"
    sched = cal.schedule.loc[:, ["open", "close"]].copy()

    # Convert to America/New_York
    sched["open_et"]  = sched["open"].dt.tz_convert("America/New_York")
    sched["close_et"] = sched["close"].dt.tz_convert("America/New_York")

    # Indicators
    #sched["is_trading_day"] = True
    sched["is_early_close"] = sched["close_et"].dt.time < pd.Timestamp("16:00", tz="America/New_York").time()

    # If you want a per-day close time (minutes since midnight ET)
    sched["session_duration"] = (sched["close_et"].dt.hour * 60 + sched["close_et"].dt.minute) - 9.5 * 60

    # Join to your intraday df by session date
    # assumes df has a Date column that is the NYSE session date (ET)
    sched_out = sched.reset_index().rename(columns={"index": "Date"})
    close_times_df = sched_out
    
    return close_times_df[['Date', 'close_et', 'is_early_close', 'session_duration']]

# Read the DBN file into a DBNStore object
dbn_store = db.DBNStore.from_file('qqq_1m.dbn')
# Convert the data to a pandas DataFrame for analysis
df = dbn_store.to_df()
df_main = df.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume']].copy()

# Add in session duration to account for early close on holidays
df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_close_times = close_times()
# Merge close times with intraday data
df_main['Date'] = pd.to_datetime(df_main['datetime_est']).dt.strftime('%Y-%m-%d')
df_close_times["Date"] = pd.to_datetime(df_close_times["Date"]).dt.date
df_main["Date"] = pd.to_datetime(df_main["Date"]).dt.date
df_main = df_main.merge(df_close_times[['Date', 'session_duration']], on="Date", how="left")

# 1. Session Structure & Market Phases

In [139]:
def add_intraday_labels(df: pd.DataFrame, dt_col: str = "datetime_est") -> pd.DataFrame:
    
    out = df.copy()

    # Ensure datetime
    out[dt_col] = pd.to_datetime(out[dt_col], errors="coerce")
    if out[dt_col].isna().any():
        bad = out[dt_col].isna().sum()
        raise ValueError(f"{bad} rows in {dt_col} could not be parsed to datetime.")

    # Extract time-of-day in minutes since midnight (ET)
    tod_minutes = out[dt_col].dt.hour * 60 + out[dt_col].dt.minute
    out["_tod_minutes"] = tod_minutes

    # Open time (09:30 ET) in minutes
    premarket_min = 7 * 60  # 420
    open_min = 9 * 60 + 30  # 570
    #close_min = 16 * 60 - 1   # 960

    # Minutes since open (can be negative pre-market, positive post-open)
    out["time_to_open"] = out["_tod_minutes"] - open_min
    out["time_to_close"] = (open_min + out["session_duration"]) - out["_tod_minutes"]

    # Column 1: simple session label
    out["session_simple"] = np.select(
        [
            out["_tod_minutes"] < premarket_min,
            (out["_tod_minutes"] >= premarket_min) & (out["_tod_minutes"] < open_min),
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] <= (open_min + out["session_duration"])),
            out["_tod_minutes"] > (open_min + out["session_duration"]),
        ],
        ["overnight", "pre_market", "open_market", "post_market"],
        default=np.nan
    )

    # Column 2: detailed session label (your buckets)
    out["session_detail"] = np.select(
        [
            # Pre-market buckets
            (out["_tod_minutes"] < 7 *60),
            (out["_tod_minutes"] >= 7*60) & (out["_tod_minutes"] < 9*60),
            (out["_tod_minutes"] >= 9*60) & (out["_tod_minutes"] < open_min),

            # Open market buckets
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] < 9*60+45),
            (out["_tod_minutes"] >= 9*60+45) & (out["_tod_minutes"] < 10*60),
            (out["_tod_minutes"] >= 10*60) & (out["_tod_minutes"] < 12*60),
            (out["_tod_minutes"] >= 12*60) & (out["_tod_minutes"] < 14*60),
            (out["_tod_minutes"] >= 14*60) & (out["_tod_minutes"] < 15*60+30),
            (out["_tod_minutes"] >= 15*60+30) & (out["_tod_minutes"] < 15*60+45),
            (out["_tod_minutes"] >= 15*60+45) & (out["_tod_minutes"] <= (open_min + out["session_duration"])),

            # Post-market buckets
            (out["_tod_minutes"] > (open_min + out["session_duration"])) & (out["_tod_minutes"] < 16*60+15),
            (out["_tod_minutes"] >= 16*60+15) & (out["_tod_minutes"] < 17*60),
            (out["_tod_minutes"] >= 17*60) & (out["_tod_minutes"] <= 20*60),
        ],
        [
            "overnight",
            "early_pre_market",
            "late_pre_market",
            "early_open",
            "late_open",
            "morning",
            "midday",
            "late_day",
            "early_close",
            "late_close",
            "early_post_market",
            "late_post_market",
            "post_market",
        ],
        default="other"
    )

    # Cleanup
    out = out.drop(columns=["_tod_minutes", "_detail_simple_check"], errors="ignore")
    return out

#df_intraday_labels[df_intraday_labels['minutes_since_open'] == 390]
#Shortest minutes_since_open = -330 largest is 629. 0 = 930am, 389 = 4:00pm
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')
df_labeled_final = df_intraday_labels[['symbol', 'datetime_est', 'time_to_open', 'time_to_close', 'session_simple', 
                    'session_detail', 'close', 'open', 'high', 'low', 'Date', 'session_duration', 'volume']].copy()

df_labeled_final

,symbol,datetime_est,time_to_open,time_to_close,session_simple,session_detail,close,open,high,low,Date,session_duration,volume
0,QQQ,2018-05-01 04:07:00-04:00,-323,713.0,overnight,overnight,160.90,160.90,160.90,160.90,2018-05-01,390.0,100
1,QQQ,2018-05-01 04:14:00-04:00,-316,706.0,overnight,overnight,160.79,160.80,160.80,160.79,2018-05-01,390.0,300
2,QQQ,2018-05-01 04:16:00-04:00,-314,704.0,overnight,overnight,160.89,160.89,160.89,160.89,2018-05-01,390.0,51
3,QQQ,2018-05-01 04:20:00-04:00,-310,700.0,overnight,overnight,160.88,160.88,160.88,160.88,2018-05-01,390.0,7
4,QQQ,2018-05-01 04:21:00-04:00,-309,699.0,overnight,overnight,160.93,160.93,160.93,160.93,2018-05-01,390.0,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1517292,QQQ,2025-12-19 18:51:00-05:00,561,-171.0,post_market,post_market,618.38,618.40,618.40,618.38,2025-12-19,390.0,157
1517293,QQQ,2025-12-19 18:55:00-05:00,565,-175.0,post_market,post_market,618.36,618.36,618.36,618.36,2025-12-19,390.0,297
1517294,QQQ,2025-12-19 18:57:00-05:00,567,-177.0,post_market,post_market,618.46,618.46,618.46,618.46,2025-12-19,390.0,20
1517295,QQQ,2025-12-19 18:58:00-05:00,568,-178.0,post_market,post_market,618.48,618.46,618.48,618.46,2025-12-19,390.0,1007


# High Level Feature Engineering

In [73]:
df_features = df_labeled_final.dropna().copy()

# OC, HL, CH, CL ratios and magnitudes
o, h, l, c = (df_features[k].to_numpy() for k in ("open", "high", "low", "close"))

def dir_th(a, b, pct):
    return (a > (1 + pct) * b).astype(np.int8) - (a < (1 - pct) * b).astype(np.int8)

pairs = {
    "OC": (c, o),
    "HL": (h, l),
    "HC": (h, c),
    "LC": (c, l),
}

for k, (a, b) in pairs.items():
    #df_features[f"{k}_Minute_Direction"] = np.sign(a - b).astype(np.int8)
    df_features[f"{k}_Minute_Magnitude"] = np.round(a / b - 1, 4)
    df_features[f"{k}_Minute_Direction_Low_TH"] = dir_th(a, b, 0.001)
    df_features[f"{k}_Minute_Direction_High_TH"] = dir_th(a, b, 0.01)

In [92]:
# Percent of winning and losing minutes per session_simple and session_detail?

def percent_direction_counts(df, column_to_count, column_to_group):

    df_counts = (
        df
        .assign(
            up   = (df[column_to_count] > 0),
            down = (df[column_to_count] < 0),
        )
        .groupby(["Date", column_to_group], sort=False)
        .agg(
            up_minutes=("up", "sum"),
            down_minutes=("down", "sum"),
        )
    ).reset_index()

    df_counts['total_minutes'] = df_counts['up_minutes'] + df_counts['down_minutes']
    df_counts['%_up_minutes'] = round(df_counts['up_minutes'] / df_counts['total_minutes'],3)
    df_counts['%_down_minutes'] = round(df_counts['down_minutes'] / df_counts['total_minutes'],3)

    return df_counts

column_to_count = 'OC_Minute_Magnitude'
column_to_group = 'session_simple'
session_simple_counts = percent_direction_counts(df_features, column_to_count, column_to_group)

column_to_count = 'OC_Minute_Magnitude'
column_to_group = 'session_detail'
session_detail_counts = percent_direction_counts(df_features, column_to_count, column_to_group)


session_detail_counts

# Anchored Time Windows

In [141]:
anchored_tw_df = df_labeled_final.copy()

In [148]:
anchored_tw_df = anchored_tw_df[anchored_tw_df['session_detail'] == 'early_open']



,symbol,datetime_est,time_to_open,time_to_close,session_simple,session_detail,close,open,high,low,Date,session_duration,volume
116,QQQ,2018-05-01 09:30:00-04:00,0,390.0,open_market,early_open,160.45,160.540,160.610,160.40,2018-05-01,390.0,214335
117,QQQ,2018-05-01 09:31:00-04:00,1,389.0,open_market,early_open,160.62,160.450,160.720,160.41,2018-05-01,390.0,46783
118,QQQ,2018-05-01 09:32:00-04:00,2,388.0,open_market,early_open,160.77,160.610,160.800,160.61,2018-05-01,390.0,52077
119,QQQ,2018-05-01 09:33:00-04:00,3,387.0,open_market,early_open,160.67,160.755,160.770,160.50,2018-05-01,390.0,29467
120,QQQ,2018-05-01 09:34:00-04:00,4,386.0,open_market,early_open,160.70,160.700,160.700,160.56,2018-05-01,390.0,38016
121,QQQ,2018-05-01 09:35:00-04:00,5,385.0,open_market,early_open,160.57,160.690,160.700,160.51,2018-05-01,390.0,82666
122,QQQ,2018-05-01 09:36:00-04:00,6,384.0,open_market,early_open,160.73,160.570,160.780,160.41,2018-05-01,390.0,84927
123,QQQ,2018-05-01 09:37:00-04:00,7,383.0,open_market,early_open,160.50,160.730,160.810,160.47,2018-05-01,390.0,46967
124,QQQ,2018-05-01 09:38:00-04:00,8,382.0,open_market,early_open,160.25,160.520,160.550,160.18,2018-05-01,390.0,80403
125,QQQ,2018-05-01 09:39:00-04:00,9,381.0,open_market,early_open,160.28,160.240,160.350,160.21,2018-05-01,390.0,70583


In [168]:
df_labeled_final[(df_labeled_final['Date'] == '2020-03-16') & (df_labeled_final['time_to_open'] <= 16)] #& (df_labeled_final['time_to_open'] > 0)]

,symbol,datetime_est,time_to_open,time_to_close,session_simple,session_detail,close,open,high,low,Date,session_duration,volume
314080,QQQ,2020-03-16 04:00:00-04:00,-330,720.0,overnight,overnight,182.01,182.01,182.01,182.01,2020-03-16,390.0,50
314081,QQQ,2020-03-16 04:01:00-04:00,-329,719.0,overnight,overnight,179.01,179.90,179.98,179.01,2020-03-16,390.0,2260
314082,QQQ,2020-03-16 04:02:00-04:00,-328,718.0,overnight,overnight,179.00,178.50,179.00,178.50,2020-03-16,390.0,249
314083,QQQ,2020-03-16 04:03:00-04:00,-327,717.0,overnight,overnight,178.92,179.01,179.32,177.50,2020-03-16,390.0,2813
314084,QQQ,2020-03-16 04:04:00-04:00,-326,716.0,overnight,overnight,179.55,179.24,179.62,179.24,2020-03-16,390.0,1016
...,...,...,...,...,...,...,...,...,...,...,...,...,...
314379,QQQ,2020-03-16 09:28:00-04:00,-2,392.0,pre_market,late_pre_market,173.99,174.33,174.33,173.99,2020-03-16,390.0,24746
314380,QQQ,2020-03-16 09:29:00-04:00,-1,391.0,pre_market,late_pre_market,173.85,173.98,174.24,173.85,2020-03-16,390.0,20366
314381,QQQ,2020-03-16 09:30:00-04:00,0,390.0,open_market,early_open,174.16,174.15,174.16,173.85,2020-03-16,390.0,4149
314382,QQQ,2020-03-16 09:45:00-04:00,15,375.0,open_market,late_open,173.00,171.50,173.00,171.26,2020-03-16,390.0,934259


In [181]:
def intraday_aggregations(df, interval):

    # ensure datetime index
    mask = (df["time_to_open"] >= 0) & (df["time_to_open"] < interval)

    daily_max_min = (
        df.loc[mask]
        .groupby("Date")["close"]
        .agg(lambda x: x.max() / x.min())
        .rename(f"max_min_{interval}m")
    )
    daily_max_min = pd.DataFrame(daily_max_min).reset_index()

    return daily_max_min


df = df_labeled_final.copy()
df_ph = pd.DataFrame()
intervals = [5, 10, 15, 30, 60]

for interval in intervals:
    print(interval)
    daily_max_min = intraday_aggregations(df, interval)

    if df_ph.empty:
        df_ph = daily_max_min.copy()
    else:
        df_ph = df_ph.merge(daily_max_min, how="left", on="Date")

    print(df_ph.columns)

df_maxmin = df_ph.copy()
df_maxmin

5
Index(['Date', 'max_min_5m'], dtype='object')
10
Index(['Date', 'max_min_5m', 'max_min_10m'], dtype='object')
15
Index(['Date', 'max_min_5m', 'max_min_10m', 'max_min_15m'], dtype='object')
30
Index(['Date', 'max_min_5m', 'max_min_10m', 'max_min_15m', 'max_min_30m'], dtype='object')
60
Index(['Date', 'max_min_5m', 'max_min_10m', 'max_min_15m', 'max_min_30m',
       'max_min_60m'],
      dtype='object')


,Date,max_min_5m,max_min_10m,max_min_15m,max_min_30m,max_min_60m
0,2018-05-01,1.001994,1.003245,1.003558,1.005493,1.007865
1,2018-05-02,1.001415,1.003451,1.003451,1.004499,1.006039
2,2018-05-03,1.001866,1.001866,1.003363,1.006415,1.008298
3,2018-05-04,1.001800,1.004532,1.005587,1.007077,1.013223
4,2018-05-07,1.001993,1.003503,1.005073,1.005797,1.005797
...,...,...,...,...,...,...
1917,2025-12-15,1.002499,1.003444,1.003721,1.006222,1.012788
1918,2025-12-16,1.001477,1.002413,1.003533,1.006925,1.008367
1919,2025-12-17,1.000735,1.003191,1.004260,1.006271,1.006271
1920,2025-12-18,1.001755,1.004012,1.005086,1.005219,1.006075
